Train Dilated CNN to predict snapshots given weak measurement outcomes of an 1D critical Ising chain.

I want to construct a machine learning model that takes weak measurement outcome as input and predicts correlation functions. The setup is as follows: Consider a critical Ising chain, H = -ZZ-X, with length L and periodic boundary condition. Consider weak measurements \propto exp(\beta \sum_{i} x_i Z_i) acting on the system where x represents the measurement outcome. I can then do projective measurement in Z basis and record the observed values of Z_i, which are recorded in a vector y. Note that each entry of x and y are either +1 or -1. I want to predict <Z_i>_x given x. 

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


# train a neural network to predict snapshots

We load the weak measurement outcomes to X and snapshot outcomes to Y. We want to build a dilated CNN to predict Y from X.

Note that each entry of X and Y is either +1 or -1, representing the measurement outcome of Z on each site for each snapshot.

The output of the NN should represent the expectation value <Z>_x for all sites. If it helps, you can convert to 0/1 labeling (rather than +1/-1) in preprocessing.

In [2]:
# load data
L = 100 # system size
beta = 0.5
X = np.load(f"../data/Fast_CritIsing_L{L}_beta{beta}_Zoutcomes_anc.npy") # shape: (N, L), where N is the number of snapshots and L is the system size
Y = np.load(f"../data/Fast_CritIsing_L{L}_beta{beta}_Zoutcomes_sys.npy") # shape: (N, L), where N is the number of snapshots and L is the system size
N = X.shape[0] # number of snapshots

# Note that each entry of X and Y is either +1 or -1, representing the measurement outcome of Z on each site for each snapshot.

In [3]:
X, Y

(array([[ 1, -1,  1, ...,  1, -1,  1],
        [ 1, -1,  1, ..., -1,  1, -1],
        [-1,  1, -1, ..., -1, -1, -1],
        ...,
        [ 1,  1,  1, ...,  1,  1,  1],
        [ 1,  1,  1, ..., -1,  1,  1],
        [ 1, -1,  1, ..., -1,  1,  1]], shape=(100000, 100)),
 array([[ 1, -1,  1, ...,  1,  1,  1],
        [ 1,  1,  1, ...,  1,  1, -1],
        [-1, -1, -1, ..., -1, -1, -1],
        ...,
        [ 1,  1,  1, ...,  1,  1,  1],
        [ 1,  1,  1, ...,  1,  1,  1],
        [ 1,  1,  1, ...,  1,  1,  1]], shape=(100000, 100)))

In [4]:
def _suggest_num_blocks(input_length, kernel_size=3, dilation_base=2):
    extra_needed = max(0.0, float(input_length - kernel_size))
    if extra_needed <= 0:
        return 1
    required_sum = extra_needed / (kernel_size - 1)
    if dilation_base == 1:
        return max(1, int(np.ceil(required_sum)))
    covered_sum = 0.0
    dilation = 1.0
    blocks = 0
    while covered_sum < required_sum:
        covered_sum += dilation
        dilation *= dilation_base
        blocks += 1
    return max(1, blocks)
def dialtedCNN(input_length, hidden_channels=64, kernel_size=3, dilation_base=2, num_blocks=None):
    if kernel_size % 2 == 0:
        raise ValueError("kernel_size must be odd to keep sequence length fixed.")
    if dilation_base < 1:
        raise ValueError("dilation_base must be >= 1.")
    if num_blocks is None:
        num_blocks = _suggest_num_blocks(
            input_length=input_length,
            kernel_size=kernel_size,
            dilation_base=dilation_base,
        )
    if num_blocks < 1:
        raise ValueError("num_blocks must be >= 1.")
    class DilatedCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.num_blocks = num_blocks
            stem_pad = (kernel_size - 1) // 2
            self.stem = nn.Conv1d(
                in_channels=1,
                out_channels=hidden_channels,
                kernel_size=kernel_size,
                padding=stem_pad,
                padding_mode="circular",
            )
            self.blocks = nn.ModuleList()
            for i in range(num_blocks):
                dilation = dilation_base ** i
                pad = dilation * (kernel_size - 1) // 2
                self.blocks.append(
                    nn.Conv1d(
                        in_channels=hidden_channels,
                        out_channels=hidden_channels,
                        kernel_size=kernel_size,
                        dilation=dilation,
                        padding=pad,
                        padding_mode="circular",
                    )
                )
            self.readout = nn.Conv1d(hidden_channels, 1, kernel_size=1)
        def forward(self, x):
            if x.dim() == 2:
                h = x.unsqueeze(1)
            elif x.dim() == 3 and x.size(1) == 1:
                h = x
            else:
                raise ValueError("Expected x shape [batch, L] or [batch, 1, L].")
            h = F.silu(self.stem(h))
            for conv in self.blocks:
                h = h + F.silu(conv(h))
            out = self.readout(h).squeeze(1)
            return torch.tanh(out)
    return DilatedCNN()


In [5]:
# preprocessing, training, and validation for a fixed system size L per run
np.random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_arr = np.asarray(X, dtype=np.float32)
Y_arr = np.asarray(Y, dtype=np.float32)
if X_arr.shape != Y_arr.shape:
    raise ValueError(f"Shape mismatch: X{X_arr.shape} vs Y{Y_arr.shape}")
if X_arr.ndim != 2:
    raise ValueError(f"Expected 2D arrays (N, L), got X with shape {X_arr.shape}")
N_samples, L_data = X_arr.shape
X_tensor = torch.from_numpy(X_arr)
Y_tensor = torch.from_numpy(Y_arr)
perm = torch.randperm(N_samples)
train_size = int(0.8 * N_samples)
train_size = max(1, min(train_size, N_samples - 1))
train_idx = perm[:train_size]
val_idx = perm[train_size:]
X_train, Y_train = X_tensor[train_idx], Y_tensor[train_idx]
X_val, Y_val = X_tensor[val_idx], Y_tensor[val_idx]
batch_size = 256
train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, Y_val), batch_size=batch_size, shuffle=False)
model = dialtedCNN(input_length=L_data, hidden_channels=64, kernel_size=3, dilation_base=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Using fixed L={L_data}, num_blocks={model.num_blocks}")
print(f"Model parameters: total={total_params:,}, trainable={trainable_params:,}")
epochs = 20
eps = 1e-6
for epoch in range(1, epochs + 1):
    model.train()
    train_mse_sum = 0.0
    train_count = 0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        # Z2 data augmentation: (X, Y) -> (-X, -Y)
        xb_aug = torch.cat([xb, -xb], dim=0)
        yb_aug = torch.cat([yb, -yb], dim=0)
        pred_m = model(xb_aug)  # in [-1, 1]
        # site-wise cross-entropy for spins in {-1, +1}
        pred_p = ((pred_m + 1.0) * 0.5).clamp(eps, 1.0 - eps)
        target_p = (yb_aug + 1.0) * 0.5
        loss = F.binary_cross_entropy(pred_p, target_p)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_mse_sum += F.mse_loss(pred_m, yb_aug, reduction="sum").item()
        train_count += yb_aug.numel()
    train_mse = train_mse_sum / train_count
    model.eval()
    val_mse_sum = 0.0
    val_count = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred_m = model(xb)
            val_mse_sum += F.mse_loss(pred_m, yb, reduction="sum").item()
            val_count += yb.numel()
    val_mse = val_mse_sum / val_count
    print(f"Epoch {epoch:03d}/{epochs} | Train MSE: {train_mse:.6f} | Val MSE: {val_mse:.6f}")


Using fixed L=100, num_blocks=6
Model parameters: total=74,433, trainable=74,433
Epoch 001/20 | Train MSE: 0.290638 | Val MSE: 0.282704
Epoch 002/20 | Train MSE: 0.282268 | Val MSE: 0.281771
Epoch 003/20 | Train MSE: 0.282022 | Val MSE: 0.281437
Epoch 004/20 | Train MSE: 0.281861 | Val MSE: 0.281211
Epoch 005/20 | Train MSE: 0.281755 | Val MSE: 0.281354
Epoch 006/20 | Train MSE: 0.281660 | Val MSE: 0.281181
Epoch 007/20 | Train MSE: 0.281710 | Val MSE: 0.281164
Epoch 008/20 | Train MSE: 0.281617 | Val MSE: 0.281191
Epoch 009/20 | Train MSE: 0.281592 | Val MSE: 0.281262
Epoch 010/20 | Train MSE: 0.281522 | Val MSE: 0.281203
Epoch 011/20 | Train MSE: 0.281547 | Val MSE: 0.281081
Epoch 012/20 | Train MSE: 0.281505 | Val MSE: 0.281189
Epoch 013/20 | Train MSE: 0.281509 | Val MSE: 0.281423
Epoch 014/20 | Train MSE: 0.281477 | Val MSE: 0.281081
Epoch 015/20 | Train MSE: 0.281439 | Val MSE: 0.281190
Epoch 016/20 | Train MSE: 0.281469 | Val MSE: 0.281194
Epoch 017/20 | Train MSE: 0.281412 | Va

In [6]:
# Save the trained model
from pathlib import Path
save_dir = Path("temp")
save_dir.mkdir(parents=True, exist_ok=True)
cnn_ckpt_path = save_dir / f"dilatedcnn_L{L_data}_beta{beta}.pt"
checkpoint = {
    "model_state_dict": model.state_dict(),
    "L": int(L_data),
    "beta": float(beta),
    "hidden_channels": 64,
    "kernel_size": 3,
    "dilation_base": 2,
    "num_blocks": int(model.num_blocks),
}
torch.save(checkpoint, cnn_ckpt_path)
print(f"Saved CNN checkpoint to: {cnn_ckpt_path}")


Saved CNN checkpoint to: temp/dilatedcnn_L100_beta0.5.pt


## Compare with logistic regression

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
import warnings
X_np = np.asarray(X, dtype=np.float32)
Y_np = np.asarray(Y, dtype=np.float32)
if X_np.shape != Y_np.shape:
    raise ValueError(f"Shape mismatch: X{X_np.shape} vs Y{Y_np.shape}")
if X_np.ndim != 2:
    raise ValueError(f"Expected 2D arrays (N, L), got X with shape {X_np.shape}")
N_samples, L_data = X_np.shape
# Reuse split from the CNN cell when available; otherwise create the same 80/20 split.
if "train_idx" in globals() and "val_idx" in globals():
    train_idx_np = train_idx.cpu().numpy() if hasattr(train_idx, "cpu") else np.asarray(train_idx)
    val_idx_np = val_idx.cpu().numpy() if hasattr(val_idx, "cpu") else np.asarray(val_idx)
else:
    rng = np.random.default_rng(0)
    perm = rng.permutation(N_samples)
    n_train = int(0.8 * N_samples)
    n_train = max(1, min(n_train, N_samples - 1))
    train_idx_np = perm[:n_train]
    val_idx_np = perm[n_train:]
X_train_lr = X_np[train_idx_np]
Y_train_lr = Y_np[train_idx_np]
X_val_lr = X_np[val_idx_np]
Y_val_lr = Y_np[val_idx_np]
Y_train_bin = ((Y_train_lr + 1.0) * 0.5).astype(np.int8)
models = []
constant_sites = 0
warnings.filterwarnings("ignore", category=ConvergenceWarning)
for site in range(L_data):
    y_site = Y_train_bin[:, site]
    if y_site.min() == y_site.max():
        models.append(float(y_site[0]))
        constant_sites += 1
        continue
    clf = LogisticRegression(solver="lbfgs", max_iter=400)
    clf.fit(X_train_lr, y_site)
    models.append(clf)
def predict_magnetization(X_eval):
    p_plus = np.empty((X_eval.shape[0], L_data), dtype=np.float32)
    for site, model_site in enumerate(models):
        if isinstance(model_site, float):
            p_plus[:, site] = model_site
        else:
            p_plus[:, site] = model_site.predict_proba(X_eval)[:, 1]
    return 2.0 * p_plus - 1.0
Y_train_pred = predict_magnetization(X_train_lr)
Y_val_pred = predict_magnetization(X_val_lr)
train_mse_lr = np.mean((Y_train_pred - Y_train_lr) ** 2)
val_mse_lr = np.mean((Y_val_pred - Y_val_lr) ** 2)
print(f"Logistic regression baseline | constant-label sites: {constant_sites}/{L_data}")
print(f"Train MSE: {train_mse_lr:.6f}")
print(f"Val MSE:   {val_mse_lr:.6f}")


Logistic regression baseline | constant-label sites: 0/100
Train MSE: 0.284344
Val MSE:   0.285570


## MSE with data size

In total we have 10^6 samples. We plot the validation MSE with 

In [ ]:
class dilatedCNNModel
    # TODO: a class wrapper for the dilated CNN model that can be used similarly as scikit-learn models.
    # should contain methods like fit(), predict(), and score().
    # should also have methods for predict_magnetization(), predict_proba()
    # should also have methods to returen the total parameters and trainable parameters of the model.
    # should also handle saving and loading the model as checkpoints.
    # When training, should have an option to also return the training history (loss and metrics per epoch) for visualization.
    # should have option to specify the random seed for reproducibility.
    # should have option to also plot validation loss if a validation set is provided.
    pass